In [1]:
from pygel3d import hmesh, jupyter_display as jd, gl_display as gl
import numpy as np
import math
from commons.utils import *
from commons.display import *
from preprocessing import voxelize
from medial_axis_formation import deep_points, postprocessing
from medial_axis_processing import unfolding, inverse_apply
from medial_axis_processing.medial_axis import MedialAxis
from scipy.spatial import KDTree

## Load data

In [2]:
input_mesh_path = '../data/x19_rough_center.ply'

input_mesh = hmesh.load(input_mesh_path)
# jd.display(input_mesh)

## Preprocess data

In [3]:
voxel_size = 1.0
smooth_steps = 5

voxelized = voxelize.voxel_remesh(input_mesh, voxel_size)
smooth(voxelized, smooth_steps)

print(len(voxelized.vertices()))
# jd.display(voxelized)

22397


## Compute Medial Axis

In [4]:
# deep_points_params = { 
#     "sigma_p": 5.0,
#     "sigma_q": 1.5,
#     "sigma_s": 4.5,
#     "omega": math.radians(90),
#     "curve_regularization_threshold": 0.95,
#     "run_sinking_smoothing": True,
#     "run_regularization": False
# } # ✌︎

# outer_points, inner_points = deep_points.deep_points(voxelized, deep_points_params)

In [5]:
# display_result(voxelize, outer_points, inner_points, show_connections=False, debug=True)

In [6]:
# save_pointset_to_file(inner_points, "results/coin_inner_points")
# save_pointset_to_file(outer_points, "results/coin_outer_points")

Extract a single sheet out of the medial sheet (the output of `deep_points.medial_sheet` is two sheets)

In [4]:
inner_points = load_pointset_from_file("results/coin_inner_points")
outer_points = load_pointset_from_file("results/coin_outer_points")

In [5]:
dihedral_angle_threshold = 45

medial_sheet = postprocessing.to_medial_sheet(voxelized, inner_points, dihedral_angle_threshold)

jd.display(medial_sheet)

FigureWidget({
    'data': [{'color': '#dddddd',
              'flatshading': False,
              'i': array([9555, 7814, 7814, ..., 9554,    0,   31]),
              'j': array([   0,    0,    1, ...,   31, 7814,    0]),
              'k': array([  31,    1, 7817, ..., 7808, 7813, 7813]),
              'type': 'mesh3d',
              'uid': 'a880097c-bf63-4fa6-899e-8bb7fc106572',
              'x': array([-22.48268283, -22.2718195 , -22.2446763 , ..., -22.44633006,
                          -23.4265175 , -23.88900754]),
              'y': array([  3.74954337,   3.71578145,   3.58990274, ..., -24.78002535,
                          -23.24734161, -23.87183123]),
              'z': array([ 0.16816882,  0.7415425 ,  0.98362398, ..., -0.61273555,  0.76223919,
                           0.58810977])},
             {'hoverinfo': 'none',
              'line': {'color': 'rgb(125,0,0)', 'width': 1},
              'mode': 'lines',
              'type': 'scatter3d',
              'uid': '792ee58

Create `MedialAxis` object from results to build correspondences between object vertices and vertices on the medial sheet

## Process Medial Axis 

In [6]:
medial_axis = MedialAxis(medial_sheet, inner_points, outer_points)

In [7]:
sheet = manifold_to_trimesh(medial_axis.mesh)
not_on_surface = [q for q in inner_points if medial_axis.inner_indices[q.index] != q.index]
not_on_surface_pos = [q.pos for q in inner_points if medial_axis.inner_indices[q.index] != q.index]

prox_query = trimesh.proximity.ProximityQuery(sheet)
_, _, face_ids = prox_query.on_surface(not_on_surface_pos)

In [8]:
for point, fid in zip(not_on_surface, face_ids):
    vid = medial_sheet.split_face_by_vertex(fid)
    medial_sheet.positions()[vid] = point.pos
    medial_axis.inner_indices[point.index] = vid

In [18]:
len(not_on_surface)

22394

In [9]:
# hmesh.triangulate(medial_sheet)
# jd.display(medial_sheet)

In [10]:
def capp(S, P, gamma, omega, N=200):
    # compute radii R, closest distance of p_i to S using KDTree for efficiency
    tree = KDTree(S)
    dist, _ = tree.query(P, k=1)
    R = dist + gamma

    # compute coverage matrix D using KDTree
    D = tree.query_ball_point(P, R)

    # Initialize Pplus=empty, Pminus=P, and a boolean mask for covered surface points
    P_plus_mask = np.zeros(len(P), dtype=bool)
    S_covered = np.zeros(len(S), dtype=bool)

    inner_points = []
    correspondences = []

    k = 0
    while not S_covered.all() and k < N:
        _coverage_scores = np.zeros(len(P))
        uniformity_scores = np.zeros(len(P))

        for i, p in enumerate(P):
            if not P_plus_mask[i]:
                _coverage_scores[i] = len([s for s in D[i] if not S_covered[s]])

                if k > 0:  # Skip for the first point
                    distances_to_selected = np.linalg.norm(P[P_plus_mask] - p, axis=1)
                    uniformity_scores[i] = np.min(distances_to_selected)

        # Normalize scores
        coverage_scores = (_coverage_scores - np.mean(_coverage_scores)) / np.std(_coverage_scores)
        if k > 0:  # Skip normalization for the first point to avoid division by zero
            uniformity_scores = (uniformity_scores - np.mean(uniformity_scores)) / np.std(uniformity_scores)

        # Combine scores to select the next point
        scores = coverage_scores + omega * uniformity_scores
        selected_index = np.argmax(scores)

        # Update the masks and selected points list
        P_plus_mask[selected_index] = True
        inner_points.append(selected_index)
        for s in D[selected_index]:
            S_covered[s] = True  # Mark surface points covered by the selected point
        correspondences.append(S[D[selected_index]])

        if k % 10 == 0:
            print(f"iter {k+1}, covered {_coverage_scores[selected_index]}")
        k += 1

    return inner_points, correspondences

In [12]:
# S = voxelized.positions()
# P = inner_points.positions
# gamma = 1.5
# omega = 1.0

# selected_medial_indices, correspondences = capp(S, P, gamma, omega)

In [12]:
# np.savetxt("selected_inner_indices", selected_medial_indices)

In [13]:
selected_medial_indices = np.loadtxt("selected_inner_indices")

In [13]:
# display_result(S, P[new_medial_pos], correspondences)

In [14]:
for q in inner_points:
    vid = medial_axis.inner_indices[q.index]
    if q.index not in selected_medial_indices:
        medial_sheet.remove_vertex(vid)
medial_sheet.cleanup()

In [16]:
len(medial_sheet.vertices())

7888

In [ ]:
jd.display(medial_sheet)

In [16]:
def display_result(outer_points, inner_points, correspondences):
    outer = go.Scatter3d(x=outer_points[:, 0],
                         y=outer_points[:, 1],
                         z=outer_points[:, 2],
                         mode='markers',
                         marker_size=1,
                         line=dict(color='rgb(125,0,0)', width=1),
                         name="surface")
    inner = go.Scatter3d(x=inner_points[:, 0],
                         y=inner_points[:, 1],
                         z=inner_points[:, 2],
                         mode='markers',
                         marker_size=3,
                         line=dict(color='rgb(0,0,125)', width=1),
                         name="inner")

    connections = []
    for i in range(len(correspondences)):
        start = inner_points[i]
        for j in range(len(correspondences[i])):
            end = correspondences[i][j]
            connections.append(start)
            connections.append(end)
            connections.append(array([None, None, None]))
    connections = array(connections)

    connecting_lines = go.Scatter3d(x=connections[:, 0],
                                    y=connections[:, 1],
                                    z=connections[:, 2],
                                    mode='lines',
                                    line=dict(color='black', width=1),
                                    hoverinfo='none',
                                    name="connections")

    fig = go.Figure(data=[outer, inner, connecting_lines])
    fig.update_layout(
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            aspectmode="data"
        ),
        width=850, height=1200
    )
    fig.show()

In [ ]:
def ball_vis(position, r):
    for i in range(r.shape[0]):
        mesh = trimesh.load('./assets/sphere_I.obj')
        mesh.vertices = mesh.vertices * r[i]
        mesh.vertices = mesh.vertices + position[i]
        mesh.export('./vis_ball/%04d.obj'%i)